# Шаблон нового частотного исследования

Здесь нет сохранённых результатов. Код генерации и обучения поддерживает произвольный диапазон k, но новый набор требует самостоятельной проверки FEM-разрешения. Все новые данные и runs записываются в отдельные каталоги. Длительные расчёты запускаются только после явного изменения параметров ниже.


In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'neural_operator').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import json
import numpy as np
import matplotlib.pyplot as plt
from neural_operator.data import load_dataset
from neural_operator.training import load_predictor, dataset_fingerprint
from neural_operator.plotting import display_table, plot_prediction
BASE = ROOT / 'neural_operator'
DATA_DIR = BASE / 'data' / 'static_low_frequency_1000x8'
MODEL_RUNS = {
    'svd': BASE / 'runs' / 'revision_2026-11-08' / 'static_low' / 'svd',
    'chebyshev': BASE / 'runs' / 'revision_2026-11-08' / 'static_low' / 'chebyshev',
    'fno': BASE / 'randomized' / 'runs' / 'low_frequency_static_1000x8' / 'fno',
}
from dataclasses import replace
from neural_operator.datasets import load_config, generate_dataset
from neural_operator.benchmark import compare_refined_fem
from neural_operator.training import TrainConfig, run_experiment
RUN_STUDY = False
RESOLUTION_ACCEPTED = False
NEW_DATA_DIR = BASE / 'data' / 'new_frequency_study'
NEW_RUNS = BASE / 'runs' / 'new_frequency_study'
NEW_CONFIG = replace(load_config(DATA_DIR), k_range=(4., 8.), h=.05)


In [ ]:
if not RUN_STUDY:
    raise RuntimeError('Set RUN_STUDY=True after choosing a new range, mesh and separate paths.')
new_dataset = generate_dataset(NEW_DATA_DIR, NEW_CONFIG)
validation = new_dataset.split_indices()[1]
selected = validation[np.argsort(new_dataset.k0[validation])[-3:]]
resolution_rows = [{'sample': int(index), **compare_refined_fem(new_dataset, int(index))}
                   for index in selected]
display_table(resolution_rows)


Сравнение с уточнённым FEM выполняется на validation-запросах до обучения сети. Примите разрешение только после отдельной оценки достаточности сетки для выбранного диапазона. Число snapshots не исправляет ошибку FEM-дискретизации.


In [ ]:
if not RESOLUTION_ACCEPTED:
    raise RuntimeError('Inspect FEM refinement and set RESOLUTION_ACCEPTED=True before training.')
for method in ('svd', 'chebyshev', 'fno'):
    config = TrainConfig(method=method, sno_architecture_version=2, pod_method='randomized')
    _, metrics, _ = run_experiment(new_dataset, NEW_RUNS / method, config)
    print(method, metrics['test_fem_l2'])


Для самостоятельного сравнения frozen и заново обученных моделей доступна `neural_operator.experiments.evaluate_frequency_shift`. Выбор рангов и параметров выполняется по validation; test используется после фиксации модели. Этот шаблон не заявляет точность на ещё не рассчитанных данных.
